# Machine Unlearning Robustness Pipeline (Kaggle)

This unified notebook allows you to test the robustness of Machine Unlearning under Quantization right here in Kaggle, utilizing the provided GPU accelerators (T4 recommended).


## 1. Environment Initialization
We first need to install the project dependencies, including `bitsandbytes` for 4-bit/8-bit quantization and Hugging Face `accelerate`.

In [1]:
# Install dependencies for Kaggle (T4 — CUDA 12.x)
# gemma3_text model type requires transformers >= 4.50.0
# Run once, then restart the kernel before proceeding.
!pip uninstall -y transformers accelerate bitsandbytes -q
!pip install -q "transformers==4.50.3" "accelerate>=0.34.0" "bitsandbytes>=0.43.0" datasets rouge-score wandb


Found existing installation: transformers 5.3.0.dev0
Uninstalling transformers-5.3.0.dev0:
  Successfully uninstalled transformers-5.3.0.dev0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
  Cloning https://github.com/huggingface/transformers (to revision v4.49.0-Gemma-3) to /tmp/pip-req-build-w1_4tp1z
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-w1_4tp1z
  Running command git checkout -q 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Resolved https://github.com/huggingface/transformers to commit 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached huggingface_hub-0.36.2-py3-none

In [2]:
import os
import sys
import warnings

# ── Suppress XLA/cuDNN/cuBLAS double-registration noise ──────────────────────
# Kaggle pre-installs TensorFlow + JAX alongside PyTorch. When CUDA plugins
# (cuFFT, cuDNN, cuBLAS) try to register twice the runtime logs harmless but
# distracting ERROR lines. These env vars silence them before any import fires.
os.environ["TF_CPP_MIN_LOG_LEVEL"]          = "3"   # 0=all, 1=info, 2=warn, 3=error off
os.environ["CUDA_DEVICE_ORDER"]             = "PCI_BUS_ID"
os.environ["XLA_FLAGS"]                     = "--xla_gpu_cuda_data_dir=/usr/local/cuda"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"]     = "true"
os.environ["GRPC_VERBOSITY"]               = "ERROR"
os.environ["GLOG_minloglevel"]              = "3"
# Suppress absl (used by XLA) before it initialises
os.environ["ABSL_LOGSINK_ALL_ENABLED"]     = "0"

# Suppress Python-level warnings from jax / tensorflow
warnings.filterwarnings("ignore", category=UserWarning, module="tensorflow")
warnings.filterwarnings("ignore", message=".*cuFFT.*")
warnings.filterwarnings("ignore", message=".*cuDNN.*")
warnings.filterwarnings("ignore", message=".*cuBLAS.*")
warnings.filterwarnings("ignore", message=".*computation_placer.*")

# ── Working directory ─────────────────────────────────────────────────────────
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    os.chdir('/kaggle/working')

print(f"Working directory: {os.getcwd()} | ON_KAGGLE={ON_KAGGLE}")


In [26]:
# --- Configuration ---
MODELS_DIR = "models"

# Flags (Set to True to force re-execution even if checkpoints exist)
FLAGS = {
    "RETRAIN_FINE_TUNE_RAW": False,
    "RETRAIN_FINE_TUNE_QA": False,
    "RETRAIN_TASK_VECTOR": True,
    "RETRAIN_GRADIENT_ASCENT": True,
    "RETRAIN_LAYERWISE_LR": True,
    "RETRAIN_GRADUAL_UNFREEZE": True,
    "RETRAIN_GRADDIFF": True,
    "REQUANTIZE": True,
    "SHORTEN_TV_TRAIN": False  # Shorten task vector train time for testing
}

CHECKPOINTS = {
    "FT_FORGET": os.path.join(MODELS_DIR, "fine_tuned_forget"),
    "FT_FORGET_QA": os.path.join(MODELS_DIR, "fine_tuned_forget_qa"),
    "TV_UNLEARN": os.path.join(MODELS_DIR, "unlearned_task_vector"),
    "TV_UNLEARN_QA": os.path.join(MODELS_DIR, "unlearned_task_vector_qa"),
    "TV_UNLEARN_BOTH": os.path.join(MODELS_DIR, "unlearned_task_vector_both"),
    "GA_UNLEARN": os.path.join(MODELS_DIR, "unlearned_gradient_ascent"),
    # Shock-reduction methods
    "LAYERWISE_LR_UNLEARN": os.path.join(MODELS_DIR, "unlearned_layerwise_lr"),
    "GRADUAL_UNFREEZE_UNLEARN": os.path.join(MODELS_DIR, "unlearned_gradual_unfreeze"),
    "GRADDIFF_UNLEARN": os.path.join(MODELS_DIR, "unlearned_graddiff"),
}

In [27]:
# import os
# import shutil

# # --- MODEL TRANSFER FROM KAGGLE INPUT TO OUTPUT ---
# # IMPORTANT: Replace 'YOUR_DATASET_NAME' with the actual folder name in Kaggle's input panel
# INPUT_SOURCE_DIR = "/kaggle/input/models/anurag2006/initial-tv-models/transformers/default/1/models" 
# OUTPUT_MODELS_DIR = MODELS_DIR # Usually resolves to /kaggle/working/models

# print(f"Checking for input models in: {INPUT_SOURCE_DIR}")

# if os.path.exists(INPUT_SOURCE_DIR):
#     # Ensure the output models directory exists
#     os.makedirs(OUTPUT_MODELS_DIR, exist_ok=True)
    
#     print(f"Transferring models to {OUTPUT_MODELS_DIR}...")
    
#     # Iterate through and copy everything from the input folder
#     for item in os.listdir(INPUT_SOURCE_DIR):
#         source_path = os.path.join(INPUT_SOURCE_DIR, item)
#         dest_path = os.path.join(OUTPUT_MODELS_DIR, item)
        
#         if os.path.isdir(source_path):
#             print(f" -> Merging/Replacing directory: {item}")
#             # dirs_exist_ok=True allows it to overwrite/merge with existing folders seamlessly
#             shutil.copytree(source_path, dest_path, dirs_exist_ok=True)
#         else:
#             print(f" -> Copying file: {item}")
#             shutil.copy2(source_path, dest_path)
            
#     print("✅ Transfer complete! Older models are now ready for inference/evaluation.")
# else:
#     print(f"⚠️ Input directory '{INPUT_SOURCE_DIR}' not found.")
#     print("If you haven't attached the dataset/model to this notebook yet, use the 'Add Data' button on the right sidebar.")

### Authentication
Tokens are hardcoded in the login cell below. To rotate them, update cell 7 directly or replace with `UserSecretsClient` calls if running on a shared notebook.

In [28]:
import os
import huggingface_hub
import wandb

# ── Tokens (read from project .env at notebook-build time) ──────────────────
HF_TOKEN    = os.getenv("HF_TOKEN", "")
WANDB_KEY   = os.getenv("WANDB_API_KEY", "")

# HuggingFace login
os.environ["HF_TOKEN"]      = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to HuggingFace.")

# Weights & Biases login
os.environ["WANDB_API_KEY"] = WANDB_KEY
wandb.login(key=WANDB_KEY, relogin=True)
print("Logged in to W&B.")
wandb.init(
    project  = "INLP-Unlearning",
    name     = "gemma3-1b-unlearning-run",
    config   = {"model": "google/gemma-3-1b-it", "dataset": "MUSE-Books"},
    resume   = "allow",
)
print("W&B run initialised.")


--- Authenticated with Hugging Face successfully! ---


In [29]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig
from datasets import load_dataset
from torch.optim import AdamW
from rouge_score import rouge_scorer

MODEL_NAME = "google/gemma-3-1b-it"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
# Suppress AccumulateGrad CUDA stream mismatch warning (harmless with grad checkpointing)
import torch.autograd
torch.autograd.graph.set_warn_on_accumulate_grad_stream_mismatch(False)


Using device: cuda


## 2. Model and Dataset Operations

In [30]:
def load_base_model(gradient_checkpointing: bool = False):
    """Load Gemma-3-1B in FP16 across available GPUs.
    gradient_checkpointing=True halves activation memory at ~33% compute cost.
    """
    print(f"Loading Base Model {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        attn_implementation="eager",  # Gemma3 recommended; sdpa triggers warnings
    )
    model.config.use_cache = False    # incompatible with gradient checkpointing
    if gradient_checkpointing:
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    return model, tokenizer

def load_muse_data():
    """
    Loads the MUSE-Books dataset.
    - 'raw' config for training (text unlearning)
    - 'knowmem' config for evaluation (QA pairs)
    """
    print("Loading MUSE-Books dataset...")
    print("Loading Raw Text (subset='raw')...")
    try:
        raw_dataset = load_dataset("muse-bench/MUSE-Books", "raw")
    except ValueError:
        print("Warning: 'raw' config not found. Fallback to default...")
        raw_dataset = load_dataset("muse-bench/MUSE-Books")

    print(f"Available 'raw' splits: {raw_dataset.keys()}")
    forget_train = raw_dataset['forget']
    retain_train = raw_dataset['retain1'] if 'retain1' in raw_dataset else raw_dataset['retain']

    print("Loading Knowledge Memorization (subset='knowmem')...")
    try:
        qa_dataset = load_dataset("muse-bench/MUSE-Books", "knowmem")
        print(f"Available 'knowmem' splits: {qa_dataset.keys()}")
        qa_forget_key = next((k for k in ['forget_qa', 'forget'] if k in qa_dataset), None)
        qa_retain_key = next((k for k in ['retain_qa', 'retain', 'retain1'] if k in qa_dataset), None)
        if qa_forget_key and qa_retain_key:
            forget_qa = qa_dataset[qa_forget_key]
            retain_qa = qa_dataset[qa_retain_key]
        else:
            print("Warning: QA keys not found. Using available splits.")
            forget_qa = qa_dataset[list(qa_dataset.keys())[0]]
            retain_qa = qa_dataset[list(qa_dataset.keys())[1]] if len(qa_dataset.keys()) > 1 else forget_qa
    except Exception as e:
        print(f"Warning: Could not load knowmem subset ({e}). Using raw as fallback.")
        forget_qa = forget_train
        retain_qa = retain_train

    print(f"Loaded: {len(forget_train)} forget / {len(retain_train)} retain train samples")
    print(f"Loaded: {len(forget_qa)} forget QA / {len(retain_qa)} retain QA pairs")
    return forget_train, retain_train, forget_qa, retain_qa

def load_tokenizer_only():
    """Load just the tokenizer without loading the 2GB model weights."""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


## 3. Unlearning Implementations
These functions define Task Arithmetic and Gradient Ascent (Baseline NPO methods).

In [31]:
def compute_task_vector(pretrained_model, finetuned_model):
    """Compute TV in FP32 on CPU (avoids VRAM pressure and FP16 rounding errors)."""
    task_vector = {}
    ft_state = finetuned_model.state_dict()
    for name, base_param in pretrained_model.named_parameters():
        if name in ft_state:
            base_f32 = base_param.detach().cpu().float()
            ft_f32   = ft_state[name].detach().cpu().float()
            task_vector[name] = ft_f32 - base_f32   # keep FP32 for precision
    return task_vector

def apply_task_vector(base_model_name_or_path, task_vector, scaling_factor):
    """Load a fresh copy of base model and apply TV in-place."""
    print("Loading fresh base model for TV application...")
    unlearned_model = AutoModelForCausalLM.from_pretrained(
        base_model_name_or_path,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        attn_implementation="eager",
    )
    with torch.no_grad():
        for name, param in unlearned_model.named_parameters():
            if name in task_vector:
                delta = task_vector[name].to(param.device).to(param.dtype)
                param.add_(delta * scaling_factor)
    return unlearned_model

In [32]:
import torch.nn.functional as F

def get_batch_logps(logits, labels):
    """Token-level log-probs, summed over non-padding positions."""
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()
    loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
    token_log_probs = -loss_fct(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1)
    ).view(shift_labels.size())
    mask = (shift_labels != -100)
    return (token_log_probs * mask).sum(dim=-1)


def npo_unlearning(model, ref_model, forget_dataloader, epochs=1, lr=1e-5, beta=0.1):
    """
    Negative Preference Optimization.
    ref_model is kept frozen; kept on a separate device if available to save VRAM.
    """
    print("Executing NPO unlearning...")
    # Enable gradient checkpointing to halve activation memory
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    model.train()
    ref_model.eval()

    # Keep ref_model on cuda:0 (or cpu). Never move to cuda:1:
    # Gemma-3 ties lm_head.weight = embed_tokens.weight; with device_map="auto"
    # embed_tokens lives on cuda:0, so a ref forward with inputs on cuda:1
    # causes a device mismatch on the lm_head linear call.
    # Follow actual ref_model device (cuda:0 when loaded with device_map={"":0}).
    ref_device = next(ref_model.parameters()).device

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler = _make_scaler()
    policy_device = next(model.parameters()).device

    for epoch in range(epochs):
        for step, batch in enumerate(forget_dataloader):
            inputs = batch['input_ids'].to(policy_device)
            masks  = batch['attention_mask'].to(policy_device)
            labels = batch['labels'].to(policy_device)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(_amp_device()):
                outputs = model(input_ids=inputs, attention_mask=masks)
                policy_logps = get_batch_logps(outputs.logits, labels)

                with torch.no_grad():
                    ref_in  = inputs.to(ref_device)
                    ref_msk = masks.to(ref_device)
                    ref_lbl = labels.to(ref_device)
                    ref_outputs = ref_model(input_ids=ref_in, attention_mask=ref_msk)
                    ref_logps = get_batch_logps(ref_outputs.logits, ref_lbl).to(policy_device)

                ratio = policy_logps - ref_logps
                loss = -(2.0 / beta) * F.logsigmoid(-beta * ratio).mean()

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"  [skip] step {step}: NaN/Inf"); continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            if step % 20 == 0:
                torch.cuda.empty_cache()
                print(f"  epoch {epoch} | step {step} | loss {loss.item():.4f}")

    return model

In [33]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

MAX_SEQ_LEN = 256  # 256 tokens: 4× less activation memory vs 1024

# ── Layer-wise LR helper (shared by fine-tune AND unlearning) ────────────────
def get_layerwise_param_groups(model, base_lr: float, decay: float = 0.85):
    """Higher LR for later (task-specific) layers, lower for earlier (general).
    Handles tied weights (Gemma-3 ties embed_tokens & lm_head) by tracking
    seen parameter ids and skipping duplicates.
    """
    try:
        transformer_layers = model.model.layers
    except AttributeError:
        return [{"params": list(model.parameters()), "lr": base_lr}]

    num_layers = len(transformer_layers)
    seen_ids   = set()
    groups     = []

    def _add_group(params, lr):
        unique = [p for p in params if id(p) not in seen_ids]
        seen_ids.update(id(p) for p in unique)
        if unique:
            groups.append({"params": unique, "lr": lr})

    emb_lr = base_lr * (decay ** num_layers)
    _add_group(list(model.model.embed_tokens.parameters()), emb_lr)
    for idx, layer in enumerate(transformer_layers):
        _add_group(list(layer.parameters()),
                   base_lr * (decay ** (num_layers - 1 - idx)))
    _add_group(list(model.lm_head.parameters()), base_lr)

    print(f"  Discriminative LR: embed={emb_lr:.2e} ... layer[-1]={base_lr:.2e} "
          f"({len(groups)} groups)")
    return groups


class DiscriminativeTrainer(Trainer):
    """Trainer with per-layer (discriminative) learning rates.

    Why fp16=False is set in TrainingArguments when using this trainer:
    The base model is loaded in torch.float16 for memory efficiency.
    Using fp16=True in Trainer on top of that causes the GradScaler to see
    FP16 gradients, which it cannot unscale (ValueError). With fp16=False,
    training runs in FP32 arithmetic on the FP16 weights — safe and correct.
    Memory is still controlled via gradient_checkpointing + paged_adamw_8bit.
    """
    def __init__(self, *args, base_lr=2e-5, lr_decay=0.85, **kwargs):
        super().__init__(*args, **kwargs)
        self._base_lr  = base_lr
        self._lr_decay = lr_decay

    def create_optimizer(self):
        param_groups = get_layerwise_param_groups(
            self.model, self._base_lr, self._lr_decay
        )
        try:
            import bitsandbytes as bnb
            if not any(p.is_cuda for g in param_groups for p in g['params']):
                raise RuntimeError('bnb requires CUDA params')
            self.optimizer = bnb.optim.PagedAdamW8bit(param_groups)
        except Exception:
            self.optimizer = torch.optim.AdamW(param_groups)
        return self.optimizer


# ── Data helpers ─────────────────────────────────────────────────────────────
def tokenize_dataset(dataset, tokenizer, max_length=MAX_SEQ_LEN):
    text_col = 'text' if 'text' in dataset.column_names else dataset.column_names[0]
    tokenized = dataset.map(
        lambda x: tokenizer(x[text_col]),
        batched=True, remove_columns=dataset.column_names, num_proc=2,
    )
    def group_texts(examples):
        concat = {k: sum(examples[k], []) for k in examples.keys()}
        total  = (len(concat[list(concat.keys())[0]]) // max_length) * max_length
        result = {k: [t[i:i+max_length] for i in range(0, total, max_length)]
                  for k, t in concat.items()}
        result["labels"] = result["input_ids"].copy()
        return result
    lm_dataset = tokenized.map(group_texts, batched=True, num_proc=2)
    lm_dataset.set_format("torch")
    print(f"  Chunked → {len(lm_dataset)} blocks of {max_length} tokens.")
    return lm_dataset

def create_dataloader(dataset, tokenizer, batch_size=2, max_length=MAX_SEQ_LEN):
    tokenized = tokenize_dataset(dataset, tokenizer, max_length)
    collator  = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    return DataLoader(tokenized, batch_size=batch_size, shuffle=True,
                      collate_fn=collator, pin_memory=True,
                      num_workers=2, drop_last=True)

def fine_tune_model(model, tokenizer, dataset, output_dir, epochs=5,
                    is_qa=False, use_discriminative_lr=False, lr_decay=0.85):
    """Fine-tune with optional discriminative (per-layer) learning rates."""
    print(f"Fine-tuning {len(dataset)} samples | epochs={epochs} | disc_lr={use_discriminative_lr}")
    if is_qa:
        def tok_qa(examples):
            return tokenizer(examples['text'], truncation=True, max_length=MAX_SEQ_LEN*4)
        tokenized_dataset = dataset.map(tok_qa, batched=True,
                                        remove_columns=dataset.column_names, num_proc=2)
    else:
        tokenized_dataset = tokenize_dataset(dataset, tokenizer)

    base_lr = 2e-5
    import os as _os
    training_args = TrainingArguments(
        output_dir=output_dir,
        run_name=f"ft-{_os.path.basename(output_dir)}",  # distinct from output_dir → no wandb warning
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=base_lr,
        weight_decay=0.01,
        save_strategy="no",
        logging_steps=10,
        fp16=False,  # model in fp16 already; Trainer fp16 causes GradScaler clash
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="paged_adamw_8bit",
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
        report_to="wandb",
        ddp_find_unused_parameters=False,
    )
    collator     = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    TrainerClass = DiscriminativeTrainer if use_discriminative_lr else Trainer
    trainer_kwargs = dict(model=model, args=training_args,
                          train_dataset=tokenized_dataset, data_collator=collator)
    if use_discriminative_lr:
        trainer_kwargs.update(base_lr=base_lr, lr_decay=lr_decay)
    trainer = TrainerClass(**trainer_kwargs)
    trainer.train()
    return model

def prepare_cloze_questions(dataset):
    questions = []
    q_col = next((c for c in ['question','prompt','input'] if c in dataset.column_names), None)
    a_col = next((c for c in ['answer','target','output'] if c in dataset.column_names), None)
    if q_col and a_col:
        for item in dataset:
            questions.append({'prompt': item[q_col], 'answer': item[a_col]})
    elif 'text' in dataset.column_names:
        for item in dataset.select(range(min(50, len(dataset)))):
            text = item['text']
            if len(text) > 100:
                questions.append({'prompt': text[:50], 'answer': text[50:60]})
    return questions

def prepare_raw_text_continuation(dataset, num_samples=50):
    pairs, text_col = [], ('text' if 'text' in dataset.column_names else dataset.column_names[0])
    for item in dataset.select(range(min(num_samples, len(dataset)))):
        words = item[text_col].split()
        if len(words) >= 50:
            pairs.append({'prompt': " ".join(words[:30]), 'answer': " ".join(words[30:50])})
    return pairs


## 4. Post-Training Quantization (PTQ)
Loading the unlearned FP16 models in 4-bit to observe the Recovery Delta.

In [34]:
def load_quantized_model(model_dir_path, bit_width=4):
    print(f"Quantizing model from {model_dir_path} to {bit_width}-bit precision...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=(bit_width == 4),
        load_in_8bit=(bit_width == 8),
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4"
    )
    
    quantized_model = AutoModelForCausalLM.from_pretrained(
        model_dir_path,
        device_map="auto",
        quantization_config=bnb_config
    )
    return quantized_model

## 5. Evaluation Loop

In [35]:
import math
import torch
from rouge_score import rouge_scorer as rouge_lib

EVAL_BATCH = 8   # how many prompts to generate in parallel during eval

def evaluate_rouge_score(model, tokenizer, cloze_questions):
    if not cloze_questions:
        return 0.0

    scorer = rouge_lib.RougeScorer(['rougeL'], use_stemmer=True)
    model.eval()
    total = 0.0

    # Batch generation: group questions into chunks of EVAL_BATCH
    for i in range(0, len(cloze_questions), EVAL_BATCH):
        batch_q = cloze_questions[i : i + EVAL_BATCH]
        prompts = [q['prompt'] for q in batch_q]
        answers = [q['answer'].lower() for q in batch_q]

        # Left-pad so all prompts align on the right
        if tokenizer.padding_side != "left":
            tokenizer.padding_side = "left"
        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128,
        )
        device = next(model.parameters()).device
        enc = enc.to(device) if hasattr(enc, 'to') else {k: v.to(device) for k, v in enc.items()}

        with torch.inference_mode():
            out = model.generate(
                **enc,
                max_new_tokens=30,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False,
            )

        # Decode only the new tokens
        input_len = enc["input_ids"].shape[1]
        for j, seq in enumerate(out):
            new_toks = seq[input_len:]
            cont = tokenizer.decode(new_toks, skip_special_tokens=True).strip().lower()
            total += scorer.score(answers[j], cont)['rougeL'].fmeasure

    # Restore right-padding for training compatibility
    tokenizer.padding_side = "right"
    return total / len(cloze_questions)


def evaluate_perplexity(model, tokenizer, dataset, num_samples=50):
    """Batched perplexity computation — much faster than one-by-one."""
    if not dataset or len(dataset) == 0:
        return float('inf')

    text_col = 'text' if 'text' in dataset.column_names else dataset.column_names[0]
    texts = [dataset[i][text_col] for i in range(min(num_samples, len(dataset)))]

    model.eval()
    total_loss, total_len = 0.0, 0

    EVAL_PPL_BATCH = 4
    for i in range(0, len(texts), EVAL_PPL_BATCH):
        batch_texts = texts[i : i + EVAL_PPL_BATCH]
        enc = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256,
        ).to(next(model.parameters()).device)

        if enc['input_ids'].size(1) < 2:
            continue

        # Build labels: -100 at pad positions so they are ignored by CE loss
        labels = enc['input_ids'].clone()
        labels[enc['attention_mask'] == 0] = -100

        with torch.inference_mode():
            out = model(
                input_ids=enc['input_ids'],
                attention_mask=enc['attention_mask'],
                labels=labels,
            )

        # Weight by real (non-padded) tokens only
        real_tokens = (labels != -100).sum().item()
        if real_tokens == 0:
            continue
        total_loss += out.loss.item() * real_tokens
        total_len  += real_tokens

    if total_len == 0:
        return float('inf')
    avg_loss = total_loss / total_len
    return math.exp(min(avg_loss, 20))

## Execution Sandbox
Run your components below to generate the metrics outlined in your proposal.

## 6. Shock-Reduction Unlearning Methods
Inverted-triangle LR, gradual unfreezing, and GradDiff to preserve retain-set performance while still unlearning the forget set.

In [ ]:
# ── AMP compat shim (CPU-safe) ───────────────────────────────────────────────
def _amp_device():
    """Returns the device string for AMP — 'cuda' if available, 'cpu' otherwise."""
    return "cuda" if torch.cuda.is_available() else "cpu"

def _make_scaler():
    """Returns a GradScaler for the current device. CPU scaler is a no-op.
    Falls back to cuda.amp for older PyTorch (<2.3).
    """
    if not torch.cuda.is_available():
        # CPU: return a dummy scaler whose methods are all no-ops
        class _NoOpScaler:
            def scale(self, loss): return loss
            def unscale_(self, opt): pass
            def step(self, opt): opt.step()
            def update(self): pass
        return _NoOpScaler()
    try: return torch.amp.GradScaler('cuda')
    except TypeError: return torch.cuda.amp.GradScaler()


# ============================================================
# Shock-Reduction Unlearning Methods
# ============================================================
import torch
import torch.nn.functional as F


# ── Manual SWA (Stochastic Weight Averaging) ─────────────────────────────────
class ManualSWA:
    """
    Lightweight SWA that works with device_map="auto" (model parallelism).
    Accumulates a running mean of parameter tensors on CPU in FP32 for
    numerical stability, then copies back to the model at the end.

    Averaging smooths out the erratic weight trajectory caused by gradient
    ascent, landing the model at a flatter, more stable "forgetting basin"
    rather than an extreme point that also hurts retain performance.
    """
    def __init__(self, model):
        self.n   = 0
        self.avg = {n: p.detach().cpu().float().clone()
                    for n, p in model.named_parameters()}

    def update(self, model):
        """Call this every swa_freq steps in the SWA phase."""
        self.n += 1
        with torch.no_grad():
            for name, param in model.named_parameters():
                cpu_param = param.detach().cpu().float()
                self.avg[name].add_((cpu_param - self.avg[name]) / self.n)

    def apply(self, model):
        """Copy averaged weights back into the model. Returns model."""
        print(f"  SWA: applying average of {self.n} snapshots...")
        with torch.no_grad():
            for name, param in model.named_parameters():
                param.copy_(self.avg[name].to(param.device).to(param.dtype))
        return model


# ── Method 1: GA with inverted-triangle LR + SWA ────────────────────────────
def ga_layerwise_lr_unlearning(
    model, forget_dataloader,
    epochs: int = 3,
    base_lr: float = 1e-5,
    decay: float = 0.85,
    grad_accum: int = 4,
    max_grad_norm: float = 1.0,
    use_swa: bool = True,
    swa_start_frac: float = 0.5,  # start collecting after this fraction of total steps
    swa_freq: int = 5,
):
    """GA with per-layer discriminative LR. SWA averages the final checkpoints
    to prevent the model landing at an extreme point on the ascent trajectory."""
    print("\n[Method] GA — inverted-triangle LR" + (" + SWA" if use_swa else ""))
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.train()

    param_groups = get_layerwise_param_groups(model, base_lr, decay)
    optimizer    = torch.optim.AdamW(param_groups)
    scaler       = _make_scaler()
    device       = next(model.parameters()).device

    total_steps  = epochs * len(forget_dataloader)
    swa_start    = int(total_steps * swa_start_frac)
    swa          = ManualSWA(model) if use_swa else None
    global_step  = 0

    for epoch in range(epochs):
        total_loss = 0.0
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(forget_dataloader):
            inputs = batch["input_ids"].to(device)
            masks  = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.amp.autocast(_amp_device()):
                out  = model(input_ids=inputs, attention_mask=masks, labels=labels)
                loss = -out.loss / grad_accum

            if not (torch.isnan(loss) or torch.isinf(loss)):
                scaler.scale(loss).backward()
            else:
                print(f"  [skip] step {step}: NaN/Inf")

            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                # SWA collection
                if use_swa and global_step >= swa_start and global_step % swa_freq == 0:
                    swa.update(model)

            total_loss += out.loss.item()
            global_step += 1

            if step % 20 == 0:
                print(f"  epoch {epoch} | step {step} | fwd-loss {out.loss.item():.4f}"
                      + (f" | swa_n={swa.n}" if use_swa and swa.n > 0 else ""))
                torch.cuda.empty_cache()

        print(f"  epoch {epoch} | avg fwd-loss {total_loss/max(1,len(forget_dataloader)):.4f}")

    if use_swa and swa.n > 0:
        swa.apply(model)
    return model


# ── Method 2: GA with gradual unfreezing + SWA ───────────────────────────────
def _freeze_all(model):
    for p in model.parameters(): p.requires_grad = False

def _unfreeze_last_n(model, n: int):
    try:
        layers = model.model.layers
    except AttributeError:
        for p in model.parameters(): p.requires_grad = True; return
    for layer in layers[-n:]:
        for p in layer.parameters(): p.requires_grad = True
    for p in model.lm_head.parameters(): p.requires_grad = True


def ga_gradual_unfreeze_unlearning(
    model, forget_dataloader,
    epochs: int = 6,
    lr: float = 1e-5,
    start_layers: int = 4,
    unfreeze_every_epoch: int = 2,
    grad_accum: int = 4,
    max_grad_norm: float = 1.0,
    use_swa: bool = True,
    swa_start_frac: float = 0.5,
    swa_freq: int = 5,
):
    """Gradual unfreezing: start with top layers, open more per phase.
    SWA on the final (fully unfrozen) phase."""
    print("\n[Method] GA — gradual unfreezing" + (" + SWA" if use_swa else ""))
    scaler = _make_scaler()
    device = next(model.parameters()).device

    try:
        num_layers = len(model.model.layers)
    except AttributeError:
        num_layers = 1

    current_unfrozen = 0
    total_steps      = epochs * len(forget_dataloader)
    swa_start        = int(total_steps * swa_start_frac)
    swa              = ManualSWA(model) if use_swa else None
    global_step      = 0

    for epoch in range(epochs):
        target = min(start_layers * (1 + epoch // unfreeze_every_epoch), num_layers)
        if target != current_unfrozen:
            _freeze_all(model)
            _unfreeze_last_n(model, target)
            current_unfrozen = target
            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  epoch {epoch}: unfreezing last {target}/{num_layers} layers "
                  f"({trainable/1e6:.1f}M trainable params)")
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

        model.train()
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=lr)
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0

        for step, batch in enumerate(forget_dataloader):
            inputs = batch["input_ids"].to(device)
            masks  = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.amp.autocast(_amp_device()):
                out  = model(input_ids=inputs, attention_mask=masks, labels=labels)
                loss = -out.loss / grad_accum

            if not (torch.isnan(loss) or torch.isinf(loss)):
                scaler.scale(loss).backward()
            else:
                print(f"  [skip] step {step}: NaN/Inf")

            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                if use_swa and global_step >= swa_start and global_step % swa_freq == 0:
                    swa.update(model)

            total_loss += out.loss.item()
            global_step += 1

            if step % 20 == 0:
                print(f"  epoch {epoch} | step {step} | fwd-loss {out.loss.item():.4f}"
                      + (f" | swa_n={swa.n}" if use_swa and swa.n > 0 else ""))
                torch.cuda.empty_cache()

        print(f"  epoch {epoch} | avg fwd-loss {total_loss/max(1,len(forget_dataloader)):.4f}")

    if use_swa and swa.n > 0:
        swa.apply(model)
    for p in model.parameters(): p.requires_grad = True
    return model


# ── Method 3: GradDiff + SWA ─────────────────────────────────────────────────
def graddiff_unlearning(
    model, forget_dataloader, retain_dataloader,
    epochs: int = 3,
    lr: float = 1e-5,
    retain_weight: float = 1.0,
    grad_accum: int = 4,
    max_grad_norm: float = 1.0,
    use_swa: bool = True,
    swa_start_frac: float = 0.5,
    swa_freq: int = 5,
):
    """loss = -CE(forget) + retain_weight*CE(retain).
    Sequential forget/retain passes keep peak activation memory = 1 batch.
    SWA smooths the final checkpoint."""
    print("\n[Method] Gradient Difference" + (" + SWA" if use_swa else ""))
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.train()
    optimizer    = torch.optim.AdamW(model.parameters(), lr=lr)
    scaler       = _make_scaler()
    device       = next(model.parameters()).device
    retain_iter  = iter(retain_dataloader)

    total_steps  = epochs * len(forget_dataloader)
    swa_start    = int(total_steps * swa_start_frac)
    swa          = ManualSWA(model) if use_swa else None
    global_step  = 0
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(epochs):
        total_fgt, total_ret = 0.0, 0.0

        for step, forget_batch in enumerate(forget_dataloader):
            # forget pass
            f_in  = forget_batch["input_ids"].to(device)
            f_msk = forget_batch["attention_mask"].to(device)
            f_lbl = forget_batch["labels"].to(device)
            with torch.amp.autocast(_amp_device()):
                f_out  = model(input_ids=f_in, attention_mask=f_msk, labels=f_lbl)
                f_loss = -f_out.loss / grad_accum
            if not (torch.isnan(f_loss) or torch.isinf(f_loss)):
                scaler.scale(f_loss).backward()

            # retain pass
            try:
                retain_batch = next(retain_iter)
            except StopIteration:
                retain_iter  = iter(retain_dataloader)
                retain_batch = next(retain_iter)
            r_in  = retain_batch["input_ids"].to(device)
            r_msk = retain_batch["attention_mask"].to(device)
            r_lbl = retain_batch["labels"].to(device)
            with torch.amp.autocast(_amp_device()):
                r_out  = model(input_ids=r_in, attention_mask=r_msk, labels=r_lbl)
                r_loss = retain_weight * r_out.loss / grad_accum
            if not (torch.isnan(r_loss) or torch.isinf(r_loss)):
                scaler.scale(r_loss).backward()

            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                if use_swa and global_step >= swa_start and global_step % swa_freq == 0:
                    swa.update(model)

            total_fgt += f_out.loss.item()
            total_ret += r_out.loss.item()
            global_step += 1

            if step % 20 == 0:
                print(f"  epoch {epoch} | step {step} | "
                      f"fgt {f_out.loss.item():.4f} | ret {r_out.loss.item():.4f}"
                      + (f" | swa_n={swa.n}" if use_swa and swa.n > 0 else ""))
                torch.cuda.empty_cache()

        n = max(1, len(forget_dataloader))
        print(f"  epoch {epoch} | avg fgt {total_fgt/n:.4f} | avg ret {total_ret/n:.4f}")

    if use_swa and swa.n > 0:
        swa.apply(model)
    return model

## 7. Pre-flight Tests
Run on CPU with a tiny mock model — catches shape bugs before wasting GPU time.

In [ ]:
# ============================================================
# Pre-flight Test Suite (runs on CPU, no GPU or HF download needed)
# ============================================================
import sys, traceback
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import DataCollatorForLanguageModeling

VOCAB, DIM, N_LAYERS, SEQ = 200, 32, 4, 16

# ── Minimal Gemma-shaped mock model ──────────────────────────────────────────
class _Layer(nn.Module):
    def __init__(self, d): super().__init__(); self.fc = nn.Linear(d, d)
    def forward(self, x, **kw): return (self.fc(x),)

class _InnerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed_tokens = nn.Embedding(VOCAB, DIM)
        self.layers = nn.ModuleList([_Layer(DIM) for _ in range(N_LAYERS)])

class MockGemma(nn.Module):
    """Minimal model matching Gemma's attribute layout (model.model.layers, lm_head)."""
    def __init__(self):
        super().__init__()
        self.model   = _InnerModel()
        self.lm_head = nn.Linear(DIM, VOCAB, bias=False)
        self.config  = type("C", (), {"_name_or_path": "mock"})()

    def forward(self, input_ids, attention_mask=None, labels=None):
        x = self.model.embed_tokens(input_ids)
        for layer in self.model.layers: x = layer(x)[0]
        logits = self.lm_head(x)
        loss   = F.cross_entropy(logits.view(-1,VOCAB), labels.view(-1)) if labels is not None else None
        return type("Out", (), {"loss": loss, "logits": logits})()

    # Must NOT override named_parameters — let nn.Module handle recurse kwarg
    def gradient_checkpointing_enable(self, **kw): pass

    @property
    def device(self): return next(self.parameters()).device


def _make_loader(n_batches=4, batch=2):
    ids  = torch.randint(1, VOCAB, (n_batches*batch, SEQ))
    mask = torch.ones_like(ids)
    lbl  = ids.clone()
    ds   = TensorDataset(ids, mask, lbl)
    def collate(items):
        a,b,c = zip(*items)
        return {"input_ids": torch.stack(a),
                "attention_mask": torch.stack(b),
                "labels": torch.stack(c)}
    return DataLoader(ds, batch_size=batch, collate_fn=collate)


PASS, FAIL = [], []

def run_test(name, fn):
    try:
        fn()
        PASS.append(name); print(f"  PASS  {name}")
    except Exception as e:
        FAIL.append(name); print(f"  FAIL  {name}: {e}")
        traceback.print_exc()

print("=" * 60)
print("Running pre-flight tests (CPU, no GPU required)...")
print("=" * 60)

# T1 — layer-wise param groups shape & monotone LR
def t_layerwise():
    m = MockGemma()
    groups = get_layerwise_param_groups(m, base_lr=1e-5, decay=0.85)
    assert len(groups) == N_LAYERS + 2, f"expected {N_LAYERS+2} groups, got {len(groups)}"
    lrs = [g["lr"] for g in groups]
    for i in range(len(lrs)-1):
        assert lrs[i] <= lrs[i+1], f"LRs not monotone: {lrs}"
run_test("layerwise_param_groups", t_layerwise)

# T2 — ManualSWA: 1 snapshot round-trip
def t_swa():
    m = MockGemma()
    swa = ManualSWA(m)
    with torch.no_grad():
        for p in m.parameters(): p.fill_(3.0)
    swa.update(m)
    assert swa.n == 1
    with torch.no_grad():
        for p in m.parameters(): p.fill_(0.0)  # corrupt model
    swa.apply(m)
    for _, p in m.named_parameters():
        diff = (p.float() - 3.0).abs().max().item()
        assert diff < 1e-3, f"SWA apply mismatch: {diff}"
run_test("manual_swa", t_swa)

# T3 — SWA running average correctness (3 snapshots → mean)
def t_swa_avg():
    m = MockGemma()
    swa = ManualSWA(m)
    for val in [1.0, 2.0, 3.0]:
        with torch.no_grad():
            for p in m.parameters(): p.fill_(val)
        swa.update(m)
    swa.apply(m)
    for _, p in m.named_parameters():
        diff = abs(p.float().mean().item() - 2.0)
        assert diff < 1e-3, f"SWA avg wrong: got {p.float().mean():.4f}"
run_test("swa_running_average", t_swa_avg)

# T4 — GA layerwise LR (CPU, 1 epoch)
def t_ga_layerwise():
    m  = MockGemma()
    dl = _make_loader(n_batches=6, batch=2)
    result = ga_layerwise_lr_unlearning(
        m, dl, epochs=1, base_lr=1e-4, decay=0.85,
        grad_accum=2, max_grad_norm=1.0,
        use_swa=True, swa_start_frac=0.4, swa_freq=2,
    )
    assert result is m
run_test("ga_layerwise_lr_unlearning", t_ga_layerwise)

# T5 — GA gradual unfreeze (CPU, 2 epochs)
def t_ga_gradual():
    m  = MockGemma()
    dl = _make_loader(n_batches=6, batch=2)
    result = ga_gradual_unfreeze_unlearning(
        m, dl, epochs=2, lr=1e-4,
        start_layers=1, unfreeze_every_epoch=1,
        grad_accum=2, max_grad_norm=1.0,
        use_swa=True, swa_start_frac=0.4, swa_freq=2,
    )
    assert result is m
    # All params restored to requires_grad=True
    assert all(p.requires_grad for p in m.parameters())
run_test("ga_gradual_unfreeze_unlearning", t_ga_gradual)

# T6 — GradDiff (CPU, 1 epoch)
def t_graddiff():
    m  = MockGemma()
    fl = _make_loader(n_batches=4, batch=2)
    rl = _make_loader(n_batches=4, batch=2)
    result = graddiff_unlearning(
        m, fl, rl, epochs=1, lr=1e-4,
        retain_weight=1.0, grad_accum=2, max_grad_norm=1.0,
        use_swa=True, swa_start_frac=0.4, swa_freq=2,
    )
    assert result is m
run_test("graddiff_unlearning", t_graddiff)

# T7 — compute_task_vector FP32 precision
def t_task_vector():
    base = MockGemma()
    ft   = MockGemma()
    with torch.no_grad():
        for (_, pb), (_, pf) in zip(base.named_parameters(), ft.named_parameters()):
            pf.copy_(pb + 0.1)
    tv = compute_task_vector(base, ft)
    for name, delta in tv.items():
        # FP32 diff — tolerance 1e-5 is safe
        err = (delta - 0.1).abs().max().item()
        assert err < 1e-5, f"TV wrong on {name}: max_err={err}"
run_test("compute_task_vector_fp32", t_task_vector)

# T8 — evaluate_rouge_score handles dict tokenizer output
def t_rouge():
    m = MockGemma()
    class TinyTok:
        pad_token, eos_token_id, padding_side = "<pad>", 1, "left"
        def __call__(self, texts, **kw):
            n = len(texts) if isinstance(texts, list) else 1
            ids = torch.randint(1, VOCAB, (n, 8))
            return {"input_ids": ids, "attention_mask": torch.ones_like(ids)}
        def decode(self, tokens, **kw): return "harry potter"
    tok = TinyTok()
    m.generate = lambda **kw: torch.randint(1, VOCAB, (kw["input_ids"].shape[0], 12))
    questions  = [{"prompt": "Who is Harry?", "answer": "harry potter"}] * 4
    score = evaluate_rouge_score(m, tok, questions)
    assert isinstance(score, float) and 0.0 <= score <= 1.0, f"bad score: {score}"
run_test("evaluate_rouge_score_dict_tok", t_rouge)

# T9 — DiscriminativeTrainer.create_optimizer builds correct param groups
#       (tested without instantiating TrainingArguments to avoid accelerate dep)
def t_disc_trainer():
    m = MockGemma()
    # Minimal mock of what Trainer stores on self
    trainer = object.__new__(DiscriminativeTrainer)
    trainer.model    = m
    trainer.args     = type("A", (), {"weight_decay": 0.01})()
    trainer._base_lr  = 2e-5
    trainer._lr_decay = 0.85
    # Manually call create_optimizer (bypasses __init__)
    opt = DiscriminativeTrainer.create_optimizer(trainer)
    assert len(opt.param_groups) == N_LAYERS + 2, (
        f"expected {N_LAYERS+2} groups, got {len(opt.param_groups)}")
    lrs = [g["lr"] for g in opt.param_groups]
    for i in range(len(lrs)-1):
        assert lrs[i] <= lrs[i+1], f"LRs not monotone: {lrs}"
run_test("DiscriminativeTrainer_optimizer", t_disc_trainer)

# T10 — freeze / unfreeze helpers
def t_freeze():
    m = MockGemma()
    _freeze_all(m)
    assert all(not p.requires_grad for p in m.parameters()), "freeze_all failed"
    _unfreeze_last_n(m, 2)
    assert any(p.requires_grad for p in m.parameters()), "unfreeze unfroze nothing"
run_test("freeze_unfreeze_helpers", t_freeze)

# T11 — _make_scaler is a no-op on CPU (no crash, loss unchanged)
def t_cpu_scaler():
    import torch.cuda
    if torch.cuda.is_available():
        return  # only meaningful on CPU
    scaler = _make_scaler()
    t = torch.tensor(2.0, requires_grad=True)
    loss = t * 3.0
    scaled = scaler.scale(loss)
    assert abs(scaled.item() - 6.0) < 1e-5, f"scale changed value: {scaled}"
    opt = torch.optim.SGD([t], lr=0.1)
    scaler.unscale_(opt)   # must not raise
    scaler.step(opt)       # must not raise
    scaler.update()        # must not raise
run_test("cpu_scaler_noop", t_cpu_scaler)

# T12 — _amp_device() returns 'cpu' when CUDA unavailable
def t_amp_device():
    dev = _amp_device()
    assert dev in ('cuda', 'cpu'), f"unexpected device: {dev}"
    if not torch.cuda.is_available():
        assert dev == 'cpu', f"expected 'cpu', got '{dev}'"
run_test("amp_device_fallback", t_amp_device)

# ── Summary ──────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print(f"Results: {len(PASS)} passed, {len(FAIL)} failed")
if FAIL:
    print("FAILED:", FAIL)
    raise RuntimeError("Pre-flight tests failed — fix before running the full pipeline.")
else:
    print("All pre-flight tests passed. Safe to run the full pipeline.")
print("=" * 60)


In [36]:
import os
import json
import shutil
import torch
import pandas as pd
from transformers import AutoModelForCausalLM
import gc

os.makedirs(MODELS_DIR, exist_ok=True)

# Helper function to clear memory aggressively
def clean_memory():
    gc.collect()
    torch.cuda.empty_cache()

# --- Execution Pipeline ---

print("\n=== PHASE 1: Loading Resources ===")
# Load Data
forget_train, retain_train, forget_qa, retain_qa = load_muse_data()

# Prepare Evaluation Data

# 1. QA Conceptual Knowledge Evaluation
forget_qa_questions = prepare_cloze_questions(forget_qa.select(range(min(50, len(forget_qa)))))
retain_qa_questions = prepare_cloze_questions(retain_qa.select(range(min(50, len(retain_qa)))))
print(f"Prepared {len(forget_qa_questions)} QA forget questions and {len(retain_qa_questions)} QA retain questions.")

# 2. Raw Text Verbatim Memorization Evaluation
forget_raw_pairs = prepare_raw_text_continuation(forget_train, num_samples=50)
retain_raw_pairs = prepare_raw_text_continuation(retain_train, num_samples=50)
print(f"Prepared {len(forget_raw_pairs)} Raw forget completions and {len(retain_raw_pairs)} Raw retain completions.")

# Load Tokenizer (Base model needed momentarily for tokenizer if not separate)
print("Loading Tokenizer...")
tokenizer = load_tokenizer_only()
clean_memory()

# --- PHASE 2: Fine-Tuning (Task Vector Prep) --- [COMMENTED OUT]
# Fine-tuning skipped — focusing on shock-reduction unlearning methods.
# Uncomment to re-enable task-vector fine-tuning.

# import os
# from transformers import AutoConfig
# def repair_checkpoint_config(ckpt_dir, base_model_name):
#     cfg_path = os.path.join(ckpt_dir, "config.json")
#     if not os.path.exists(ckpt_dir):
#         return
#     try:
#         with open(cfg_path) as f:
#             cfg_data = json.load(f)
#         if "model_type" not in cfg_data:
#             raise ValueError("missing model_type")
#     except Exception:
#         print(f"  Repairing config in {ckpt_dir}...")
#         AutoConfig.from_pretrained(base_model_name).save_pretrained(ckpt_dir)
#         print(f"  Done.")
#
# for ckpt in [CHECKPOINTS["FT_FORGET"], CHECKPOINTS["FT_FORGET_QA"]]:
#     repair_checkpoint_config(ckpt, MODEL_NAME)
#
# print("\n=== PHASE 2: Fine-Tuning (Task Vector Prep) ===")
# if not os.path.exists(CHECKPOINTS["FT_FORGET"]) or FLAGS["RETRAIN_FINE_TUNE_RAW"]:
#     print("Starting Fine-Tuning on Forget Set (Raw Text)...")
#     base_model, _ = load_base_model()
#     tv_epochs = 2 if FLAGS.get("SHORTEN_TV_TRAIN", False) else 10
#     print(f"Task Vector Fine-Tuning Epochs set to: {tv_epochs}")
#     ft_model = fine_tune_model(base_model, tokenizer, forget_train, CHECKPOINTS["FT_FORGET"], epochs=tv_epochs,
#     use_discriminative_lr=True, lr_decay=0.85)
#     ft_model.save_pretrained(CHECKPOINTS["FT_FORGET"])
#     tokenizer.save_pretrained(CHECKPOINTS["FT_FORGET"])
#     ft_model.config.save_pretrained(CHECKPOINTS["FT_FORGET"])
#     del base_model, ft_model
#     clean_memory()
# else:
#     print(f"Checkpoint found: {CHECKPOINTS['FT_FORGET']}")

# --- PHASE 2.1: Fine-Tuning (Task Vector Prep - QA) --- [COMMENTED OUT]
# print("\n=== PHASE 2.1: Fine-Tuning (Task Vector Prep - QA) ===")
# if not os.path.exists(CHECKPOINTS["FT_FORGET_QA"]) or FLAGS["RETRAIN_FINE_TUNE_QA"]:
#     base_model, _ = load_base_model()
#     def format_qa_for_training(example):
#         q_cols = ['question', 'prompt', 'input']
#         a_cols = ['answer', 'target', 'output']
#         q_col = next((col for col in q_cols if col in example.keys()), None)
#         a_col = next((col for col in a_cols if col in example.keys()), None)
#         if q_col and a_col:
#             messages = [{"role": "user", "content": example[q_col]},
#                         {"role": "assistant", "content": example[a_col]}]
#             return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}
#         return {"text": str(example)}
#     forget_qa_train = forget_qa.map(format_qa_for_training)
#     tv_qa_epochs = 5 if FLAGS.get("SHORTEN_TV_TRAIN", False) else 20
#     ft_model_qa = fine_tune_model(base_model, tokenizer, forget_qa_train, CHECKPOINTS["FT_FORGET_QA"],
#         epochs=tv_qa_epochs, is_qa=True, use_discriminative_lr=True, lr_decay=0.85)
#     ft_model_qa.save_pretrained(CHECKPOINTS["FT_FORGET_QA"])
#     tokenizer.save_pretrained(CHECKPOINTS["FT_FORGET_QA"])
#     ft_model_qa.config.save_pretrained(CHECKPOINTS["FT_FORGET_QA"])
#     del base_model, ft_model_qa
#     clean_memory()
# else:
#     print(f"Checkpoint found: {CHECKPOINTS['FT_FORGET_QA']}")

# --- PHASE 3/3.1/3.2: Task Vector Unlearning --- [COMMENTED OUT]
# Requires fine-tuned checkpoints from Phase 2/2.1. Uncomment together with those phases.
# print("\n=== PHASE 3: Task Vector Unlearning ===")
# ... (see git history or uncomment Phase 2 first)

# # --- PHASE 4: NPO Unlearning ---
# print("\n=== PHASE 4: NPO Unlearning ===")
# if True: # Always run NPO for now as per user request to retry/fix
# # if not os.path.exists(CHECKPOINTS["GA_UNLEARN"]) or FLAGS["RETRAIN_GRADIENT_ASCENT"]:
#     print("Executing NPO Unlearning...")
#     # Load the base model to use as the frozen reference
#     ref_model, _ = load_base_model() 
    
#     # Load a fresh copy of the base model to apply unlearning to
#     unlearning_model, _ = load_base_model() 
    
#     # Create DataLoader using new function (chunks text into 512 tokens -> many batches)
#     forget_loader = create_dataloader(forget_train, tokenizer, batch_size=2)
    
#     # Run NPO (beta=0.1)
#     # Increased epochs to 5 for better convergence
#     ga_unlearned_model = npo_unlearning(
#         unlearning_model, 
#         ref_model, 
#         forget_loader, 
#         epochs=5, 
#         lr=1e-5, 
#         beta=0.1
#     )
    
#     print(f"Saving NPO Unlearned Model to {CHECKPOINTS['GA_UNLEARN']}...")
#     ga_unlearned_model.save_pretrained(CHECKPOINTS["GA_UNLEARN"])
#     tokenizer.save_pretrained(CHECKPOINTS["GA_UNLEARN"])
    
#     del ref_model, unlearning_model, ga_unlearned_model
#     clean_memory()
# else:
#     print(f"Checkpoint found: {CHECKPOINTS['GA_UNLEARN']}")

# --- PHASE 5: Evaluation & Quantization ---
print("\n=== PHASE 5: Evaluation & Quantization ===")
results = []

def run_evaluation(model_obj_or_path, model_name, is_quantized=False):
    print(f"Evaluating {model_name}...")
    try:
        model = None
        if isinstance(model_obj_or_path, str):
            if is_quantized:
                # Load bitsandbytes quantized
                try:
                    from transformers import BitsAndBytesConfig
                    bnb_config = BitsAndBytesConfig(
                        load_in_4bit=True,
                        bnb_4bit_compute_dtype=torch.float16
                    )
                    model = AutoModelForCausalLM.from_pretrained(
                        model_obj_or_path,
                        quantization_config=bnb_config,
                        device_map="auto",
                        config=AutoConfig.from_pretrained(MODEL_NAME),
                    )
                except Exception as e:
                    print(f"Quantization failed: {e}. Loading FP16 instead.")
                    model = AutoModelForCausalLM.from_pretrained(model_obj_or_path, torch_dtype=torch.float16, device_map="auto")
            else:
                model = AutoModelForCausalLM.from_pretrained(model_obj_or_path, torch_dtype=torch.float16, device_map="auto")
        else:
            model = model_obj_or_path

        forget_qa_rouge = evaluate_rouge_score(model, tokenizer, forget_qa_questions)
        retain_qa_rouge = evaluate_rouge_score(model, tokenizer, retain_qa_questions)
        
        forget_raw_rouge = evaluate_rouge_score(model, tokenizer, forget_raw_pairs)
        retain_raw_rouge = evaluate_rouge_score(model, tokenizer, retain_raw_pairs)
        
        forget_ppl = evaluate_perplexity(model, tokenizer, forget_train, num_samples=30)
        retain_ppl = evaluate_perplexity(model, tokenizer, retain_train, num_samples=30)
        
        # Helper cleanup
        if isinstance(model_obj_or_path, str):
            del model
            clean_memory()
            
        return {
            "Model": model_name,
            "Forget QA ROUGE": forget_qa_rouge,
            "Retain QA ROUGE": retain_qa_rouge,
            "Forget Raw ROUGE": forget_raw_rouge,
            "Retain Raw ROUGE": retain_raw_rouge,
            "Forget PPL": forget_ppl,
            "Retain PPL": retain_ppl
        }
    except Exception as e:
        import traceback
        print(f"ERROR evaluating {model_name}: {e}")
        traceback.print_exc()
        nan = float("nan")  # use NaN not string so DataFrame stays numeric
        return {
            "Model": model_name,
            "Forget QA ROUGE": nan,
            "Retain QA ROUGE": nan,
            "Forget Raw ROUGE": nan,
            "Retain Raw ROUGE": nan,
            "Forget PPL": nan,
            "Retain PPL": nan,
            "Error": str(e),
        }

# 1. Base Model
print("Evaluating Base Model...")
base_model, _ = load_base_model()
results.append(run_evaluation(base_model, "Base Model (FP16)"))
del base_model
clean_memory()

# 2. Unlearned Models (FP16 & 4-bit)
# TV checkpoints commented out (require Phase 2/3 fine-tuning)
checkpoints = [
    # ("TV Unlearned (Raw)", CHECKPOINTS["TV_UNLEARN"]),
    # ("TV Unlearned (QA)", CHECKPOINTS["TV_UNLEARN_QA"]),
    # ("TV Unlearned (Combined)", CHECKPOINTS["TV_UNLEARN_BOTH"]),
    # ("GA Unlearned", CHECKPOINTS["GA_UNLEARN"]),
]

for name, path in checkpoints:
    if os.path.exists(path):
        # FP16 Evaluation
        results.append(run_evaluation(path, f"{name} (FP16)", is_quantized=False))
        # 4-bit Quantization & Evaluation
        if FLAGS["REQUANTIZE"]:
             results.append(run_evaluation(path, f"{name} (4-bit)", is_quantized=True))

# --- Summary ---
print("\n=== Final Results ===")
df_results = pd.DataFrame(results)
print(df_results)
if not df_results.empty:
    df_results.to_csv("unlearning_benchmark_results.csv", index=False)
# Warn loudly if any evaluations failed
import math
failed_evals = [r["Model"] for r in results if any(
    isinstance(v, float) and math.isnan(v) for v in r.values() if isinstance(v, (int, float))
)]
if failed_evals:
    print(f"WARNING: {len(failed_evals)} evaluation(s) produced NaN metrics: {failed_evals}")
    print("Check the traceback output above for each failed model.")
else:
    print("Pipeline Successfully Completed — all evaluations produced valid metrics.")



=== PHASE 1: Loading Resources ===
Loading MUSE-Books dataset...
Loading Raw Text (subset='raw')...
Available 'raw' splits: dict_keys(['retain2', 'forget', 'retain1', 'holdout'])
Loading Knowledge Memorization (subset='knowmem')...
Available 'knowmem' splits: dict_keys(['retain_qa_icl', 'retain_qa', 'forget_qa', 'forget_qa_icl'])
Loaded Training Data: 4 forget samples, 12 retain samples.
Loaded Evaluation Data: 100 forget QA pairs, 100 retain QA pairs.
Prepared 50 QA forget questions and 50 QA retain questions.
Prepared 4 Raw forget completions and 12 Raw retain completions.
Loading Base Model Tokenizer...
Loading Base Model google/gemma-3-1b-it...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]


=== PHASE 2: Fine-Tuning (Task Vector Prep) ===
Checkpoint found: models/fine_tuned_forget

=== PHASE 2.1: Fine-Tuning (Task Vector Prep - QA) ===
Checkpoint found: models/fine_tuned_forget_qa

=== PHASE 3: Task Vector Unlearning ===
Computing Task Vector...
Loading Base Model google/gemma-3-1b-it...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

...

Evaluating TV Unlearned (Combined) (FP16)...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Evaluating TV Unlearned (Combined) (4-bit)...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]


=== Final Results ===
                             Model  Forget QA ROUGE  Retain QA ROUGE  \
0                Base Model (FP16)         0.038279         0.043943   
1        TV Unlearned (Raw) (FP16)         0.030729         0.029696   
2       TV Unlearned (Raw) (4-bit)         0.038589         0.019561   
3         TV Unlearned (QA) (FP16)         0.017782         0.005439   
4        TV Unlearned (QA) (4-bit)         0.015350         0.012238   
5   TV Unlearned (Combined) (FP16)         0.028417         0.022971   
6  TV Unlearned (Combined) (4-bit)         0.027946         0.019679   

   Forget Raw ROUGE  Retain Raw ROUGE   Forget PPL  Retain PPL  
0          0.135306          0.068283    28.168909   28.381639  
1          0.084884          0.059806  3671.845230  312.690333  
2          0.153432          0.031015   475.472546  268.739591  
3          0.148496          0.034572   160.955665  544.440787  
4          0.088470          0.045720   226.170434  652.366445  
5         

### Shock-Reduction Execution Phases
Run after the standard TV pipeline (Phases 1–5) since evaluation `results` list must already exist.

In [ ]:
# ============================================================
# Shock-Reduction Unlearning Pipeline Phases
# ============================================================

# --- PHASE A: GA with Inverted-Triangle (Layer-wise) LR ---
print("\n=== PHASE A: GA with Inverted-Triangle LR ===")
if not os.path.exists(CHECKPOINTS["LAYERWISE_LR_UNLEARN"]) or FLAGS["RETRAIN_LAYERWISE_LR"]:
    model_a, _ = load_base_model(gradient_checkpointing=True)
    forget_loader = create_dataloader(forget_train, tokenizer, batch_size=2)

    model_a = ga_layerwise_lr_unlearning(
        model_a, forget_loader,
        epochs=3, base_lr=1e-5, decay=0.85, grad_accum=4,
    )
    model_a.save_pretrained(CHECKPOINTS["LAYERWISE_LR_UNLEARN"])
    tokenizer.save_pretrained(CHECKPOINTS["LAYERWISE_LR_UNLEARN"])
    del model_a, forget_loader
    clean_memory()
else:
    print(f"Checkpoint found: {CHECKPOINTS['LAYERWISE_LR_UNLEARN']}")


# --- PHASE B: GA with Gradual Unfreezing ---
print("\n=== PHASE B: GA with Gradual Unfreezing ===")
if not os.path.exists(CHECKPOINTS["GRADUAL_UNFREEZE_UNLEARN"]) or FLAGS["RETRAIN_GRADUAL_UNFREEZE"]:
    model_b, _ = load_base_model()   # gradient_checkpointing toggled per-epoch inside
    forget_loader = create_dataloader(forget_train, tokenizer, batch_size=2)

    model_b = ga_gradual_unfreeze_unlearning(
        model_b, forget_loader,
        epochs=6, lr=1e-5,
        start_layers=4, unfreeze_every_epoch=2, grad_accum=4,
    )
    model_b.save_pretrained(CHECKPOINTS["GRADUAL_UNFREEZE_UNLEARN"])
    tokenizer.save_pretrained(CHECKPOINTS["GRADUAL_UNFREEZE_UNLEARN"])
    del model_b, forget_loader
    clean_memory()
else:
    print(f"Checkpoint found: {CHECKPOINTS['GRADUAL_UNFREEZE_UNLEARN']}")


# --- PHASE C: GradDiff ---
print("\n=== PHASE C: Gradient Difference (GradDiff) ===")
if not os.path.exists(CHECKPOINTS["GRADDIFF_UNLEARN"]) or FLAGS["RETRAIN_GRADDIFF"]:
    model_c, _ = load_base_model(gradient_checkpointing=True)
    forget_loader  = create_dataloader(forget_train, tokenizer, batch_size=2)
    retain_loader  = create_dataloader(retain_train, tokenizer, batch_size=2)

    model_c = graddiff_unlearning(
        model_c, forget_loader, retain_loader,
        epochs=3, lr=1e-5, retain_weight=1.0, grad_accum=4,
    )
    model_c.save_pretrained(CHECKPOINTS["GRADDIFF_UNLEARN"])
    tokenizer.save_pretrained(CHECKPOINTS["GRADDIFF_UNLEARN"])
    del model_c, forget_loader, retain_loader
    clean_memory()
else:
    print(f"Checkpoint found: {CHECKPOINTS['GRADDIFF_UNLEARN']}")


# --- Evaluate shock-reduction methods ---
print("\n=== Evaluating Shock-Reduction Methods ===")
shock_checkpoints = [
    ("GA LayerLR",         CHECKPOINTS["LAYERWISE_LR_UNLEARN"]),
    ("GA GradualUnfreeze", CHECKPOINTS["GRADUAL_UNFREEZE_UNLEARN"]),
    ("GradDiff",           CHECKPOINTS["GRADDIFF_UNLEARN"]),
]
for name, path in shock_checkpoints:
    if os.path.exists(path):
        results.append(run_evaluation(path, f"{name} (FP16)", is_quantized=False))
        if FLAGS["REQUANTIZE"]:
            results.append(run_evaluation(path, f"{name} (4-bit)", is_quantized=True))

print("\n=== Updated Final Results ===")
df_results = pd.DataFrame(results)
print(df_results.to_string())
df_results.to_csv("unlearning_benchmark_results.csv", index=False)
print("Shock-reduction phases complete!")

wandb.finish()
print("W&B run finished — all phases complete.")
